In [3]:
import torch
import pandas as pd
import numpy as np
from utils import DATA_DIR, select_gpu
from sentence_transformers import SentenceTransformer
from tucker_tensor import TuckerDecomposition

# if there is already a device selected on a previous execution of this cell, it should be kept
# we check if there is a global "device" variable already defined, if not we run the cell
try:
    device
except NameError:
    device = select_gpu()


def unique_pairwise_cosine(embeddings, labels=None):
    """
    embeddings: array of shape (n, d)
    labels: list of length n (optional)
    """
    n = embeddings.shape[0]
    if labels is None:
        labels = list(range(n))

    norms = np.linalg.norm(embeddings, axis=1)
    results = []

    for i in range(n):
        for j in range(i + 1, n):
            sim = np.dot(embeddings[i], embeddings[j]) / (norms[i] * norms[j])
            results.append({
                "i": labels[i],
                "j": labels[j],
                "similarity": sim
            })

    return pd.DataFrame(results)

def unique_pairwise_cosine_tucker(tucker_projections, labels=None):
    n = len(tucker_projections)
    if labels is None:
        labels = list(range(n))

    results = []
    norms = [np.linalg.norm(p.flatten()) for p in tucker_projections]

    for i in range(n):
        p1 = tucker_projections[i].flatten()
        for j in range(i + 1, n):
            p2 = tucker_projections[j].flatten()
            sim = np.dot(p1, p2) / (norms[i] * norms[j])
            results.append({
                "i": labels[i],
                "j": labels[j],
                "similarity": sim
            })

    return pd.DataFrame(results)



model = SentenceTransformer("NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers")

Selected GPU 1 with 454.62 MB used memory.


In [4]:
sentences = [
    "De soldaat vecht in het leger",
    "De president voert oorlog",
    "De kok kookt soep",
    "De kok verbrandt de soep",
    "Het kind tekent een dier",
    "Het kind schildert een dier",
    "De hond achtervolgt de kat",
    "De hond bewaakt het huis",
    "De vrouw koopt vis",
    "De vrouw wast vis",
    "De jongen eet ontbijt",
    "Hij verslindt zijn lunch",
    "Hij leest graag boeken",
    "Hij verslindt boeken",
    "Het meisje speelt voetbal",
    "De jongen speelt voetbal",
    "Het meisje eet ontbijt",
    # # GPT ADDITIONs
    # # Beroepen
    # "De leraar legt de stof uit",
    # "De leraar corrigeert het huiswerk",
    # "De arts onderzoekt de patiënt",
    # "De arts behandelt de patiënt",
    # "De timmerman bouwt een kast",
    # "De timmerman zaagt een plank",
    # "De schrijver schrijft een verhaal",
    # "De schrijver herschrijft het verhaal",
    #
    # # Dagelijkse activiteiten
    # "De man drinkt koffie",
    # "De man zet koffie",
    # "De vrouw leest een krant",
    # "De vrouw vouwt de krant",
    # "De jongen opent de deur",
    # "De jongen sluit de deur",
    # "Het meisje pakt een appel",
    # "Het meisje schilt een appel",
    #
    # # Dieren
    # "De kat jaagt op een muis",
    # "De kat vangt een muis",
    # "De vogel bouwt een nest",
    # "De vogel voedt zijn jongen",
    # "Het paard trekt de kar",
    # "Het paard draagt een ruiter",
    #
    # # Objecten
    # "De robot tilt een doos",
    # "De robot sorteert dozen",
    # "De auto raakt een paal",
    # "De auto vervoert passagiers",
    #
    # # Cognitief
    # "De student begrijpt de theorie",
    # "De student onthoudt de theorie",
    # "De onderzoeker bestudeert het fenomeen",
    # "De onderzoeker analyseert het fenomeen",
]


sentence_tuples = [
    ("vechten", "soldaat", "leger"),
    ("voeren", "president", "oorlog"),
    ("koken", "kok", "soep"),
    ("verbranden", "kok", "soep"),
    ("tekenen", "kind", "dier"),
    ("schilderen", "kind", "dier"),
    ("achtervolgen", "hond", "kat"),
    ("bewaken", "hond", "huis"),
    ("kopen", "vrouw", "vis"),
    ("wassen", "vrouw", "vis"),
    ("eten", "jongen", "ontbijt"),
    ("verslinden", "hij", "lunch"),
    ("lezen", "hij", "boek"),
    ("verslinden", "hij", "boek"),
    ("spelen", "meisje", "voetbal"),
    ("spelen", "jongen", "voetbal"),
    ("eten", "meisje", "ontbijt"),
    # # GPT ADDITIONs
    # # Beroepen
    # ("uitleggen", "leraar", "stof"),
    # ("corrigeren", "leraar", "huiswerk"),
    # ("onderzoeken", "arts", "patiënt"),
    # ("behandelen", "arts", "patiënt"),
    # ("bouwen", "timmerman", "kast"),
    # ("zagen", "timmerman", "plank"),
    # ("schrijven", "schrijver", "verhaal"),
    # ("herschrijven", "schrijver", "verhaal"),
    #
    # # Dagelijkse activiteiten
    # ("drinken", "man", "koffie"),
    # ("zetten", "man", "koffie"),
    # ("lezen", "vrouw", "krant"),
    # ("vouwen", "vrouw", "krant"),
    # ("openen", "jongen", "deur"),
    # ("sluiten", "jongen", "deur"),
    # ("pakken", "meisje", "appel"),
    # ("schillen", "meisje", "appel"),
    #
    # # Dieren
    # ("jagen", "kat", "muis"),
    # ("vangen", "kat", "muis"),
    # ("bouwen", "vogel", "nest"),
    # ("voeden", "vogel", "jongen"),
    # ("trekken", "paard", "kar"),
    # ("dragen", "paard", "ruiter"),
    #
    # # Objecten
    # ("tillen", "robot", "doos"),
    # ("sorteren", "robot", "dozen"),
    # ("raken", "auto", "paal"),
    # ("vervoeren", "auto", "passagier"),
    #
    # # Cognitief
    # ("begrijpen", "student", "theorie"),
    # ("onthouden", "student", "theorie"),
    # ("bestuderen", "onderzoeker", "fenomeen"),
    # ("analyseren", "onderzoeker", "fenomeen"),
]

In [5]:
tensor = TuckerDecomposition.load_from_disk(
    dataset="fineweb_sparse",
    method="sc",
    dims=5000,
    rank=100,
    iterations=1000,
    map_location="cpu"
)
# we check if all sentence tuples can be recognized by the tensor model
for sent in sentence_tuples:
    try:
        tensor.fetch_latents(sent)
    except KeyError as e:
        print(f"Tuple {sent} contains out-of-vocabulary word: {e}")

In [6]:
embeddings = model.encode(sentences)
projections = [tensor.contribution_tensor(sent_tuple) for sent_tuple in sentence_tuples]

In [7]:
df_unique = unique_pairwise_cosine(embeddings, labels=sentences)
df_proj_unique = unique_pairwise_cosine_tucker(projections, labels=sentences)

In [8]:
# we check for rank correlation between the two similarity measures
from scipy.stats import spearmanr
merged = pd.merge(
    df_unique,
    df_proj_unique,
    on=["i", "j"],
    suffixes=("_sbert", "_tucker")
)
corr, p_value = spearmanr(merged["similarity_sbert"], merged["similarity_tucker"])
print(f"Spearman correlation: {corr}, p-value: {p_value}")

Spearman correlation: 0.24065760535365396, p-value: 0.004769213569725169


In [9]:
for dim in [1000, 1500, 2000, 2500, ]: #5000]:
    for method in ["counting", "sc"]:
        try:
            tensor = TuckerDecomposition.load_from_disk(
                dataset="fineweb_sparse",
                method=method,
                dims=dim,
                rank=150,
                iterations=10000,
                map_location="cpu"
            )
            print("10K iterations loaded")
        except:
            tensor = TuckerDecomposition.load_from_disk(
                dataset="fineweb_sparse",
                method=method,
                dims=dim,
                rank=150,
                iterations=1000,
                map_location="cpu"
            )
            print("1K iterations loaded")
        sentences_subset = []
        sentence_tuples_subset = []
        for (tuple, sent) in zip(sentence_tuples, sentences):
            try:
                tensor.fetch_latents(tuple)
                sentences_subset.append(sent)
                sentence_tuples_subset.append(tuple)
            except KeyError as e:
                continue
        print(f"Dimension: {dim}, Method: {method} - Using {len(sentences_subset)} sentences out of {len(sentences)}")
        embeddings = model.encode(sentences_subset)
        df_unique = unique_pairwise_cosine(embeddings, labels=sentences_subset)
        projections = [tensor.contribution_tensor(sent_tuple) for sent_tuple in sentence_tuples_subset]
        df_proj_unique = unique_pairwise_cosine_tucker(projections, labels=sentences)
        merged = pd.merge(
            df_unique,
            df_proj_unique,
            on=["i", "j"],
            suffixes=("_sbert", f"_{method}_tucker")
        )
        corr, p_value = spearmanr(merged["similarity_sbert"], merged[f"similarity_{method}_tucker"])
        print(f"Dimension: {dim}, Method: {method} - Spearman correlation: {corr}, p-value: {p_value}")

10K iterations loaded
Dimension: 1000, Method: counting - Using 9 sentences out of 17
Dimension: 1000, Method: counting - Spearman correlation: 0.3939393939393939, p-value: 0.25999776683488757
10K iterations loaded
Dimension: 1000, Method: sc - Using 9 sentences out of 17
Dimension: 1000, Method: sc - Spearman correlation: 0.6833490732084478, p-value: 0.02937668347961381
10K iterations loaded
Dimension: 1500, Method: counting - Using 13 sentences out of 17
Dimension: 1500, Method: counting - Spearman correlation: -0.013058541261488244, p-value: 0.9321517954557161
10K iterations loaded
Dimension: 1500, Method: sc - Using 13 sentences out of 17
Dimension: 1500, Method: sc - Spearman correlation: 0.155933630307961, p-value: 0.30636795893809493
10K iterations loaded
Dimension: 2000, Method: counting - Using 13 sentences out of 17
Dimension: 2000, Method: counting - Spearman correlation: -0.009552356802178044, p-value: 0.9503419854323707
10K iterations loaded
Dimension: 2000, Method: sc - U

In [10]:
import tucker_tensor
from tucker_tensor import TuckerDecomposition
for dim in [1000, 2000, 3000]:# 5000]:
    for method in ["sc"]:
        tensor = TuckerDecomposition.load_from_disk(
            dataset="fineweb_large",
            method=method,
            dims=dim,
            rank=150,
            iterations=2000,
            map_location="cpu",
            name="sim"
        )
        print("semantic stopping criterium: iterations loaded")
        sentences_subset = []
        sentence_tuples_subset = []
        for (tuple, sent) in zip(sentence_tuples, sentences):
            try:
                tensor.fetch_latents(tuple)
                sentences_subset.append(sent)
                sentence_tuples_subset.append(tuple)
            except KeyError as e:
                continue
        print(f"Dimension: {dim}, Method: {method} - Using {len(sentences_subset)} sentences out of {len(sentences)}")
        embeddings = model.encode(sentences_subset)
        df_unique = unique_pairwise_cosine(embeddings, labels=sentences_subset)
        projections = [tensor.contribution_tensor(sent_tuple) for sent_tuple in sentence_tuples_subset]
        df_proj_unique = unique_pairwise_cosine_tucker(projections, labels=sentences)
        merged = pd.merge(
            df_unique,
            df_proj_unique,
            on=["i", "j"],
            suffixes=("_sbert", f"_{method}_tucker")
        )
        corr, p_value = spearmanr(merged["similarity_sbert"], merged[f"similarity_{method}_tucker"])
        print(f"Dimension: {dim}, Method: {method} - Spearman correlation: {corr}, p-value: {p_value}")

Loaded Tucker decomposition with the following parameters:
  timestamp: 2025-12-08T10:52:44.339772Z
  dataset: fineweb_dutch
  input_vectors: 10000000
  method: sc
  dimensionality: 1000
  rank: [150, 150, 150]
  max_iterations: 2000
  iterations: 899
  final_error: 0.9388959407806396
  final_fitness: 0.0188480741791619
  tolerance: 1e-06
  random_state: 1
  output_tensor: sim_sc_1000d_150r_2000i.pt
  runtime_seconds: 1442.09
semantic stopping criterium: iterations loaded
Dimension: 1000, Method: sc - Using 9 sentences out of 17
Dimension: 1000, Method: sc - Spearman correlation: 0.8896375227527807, p-value: 0.0005669647218379624
Loaded Tucker decomposition with the following parameters:
  timestamp: 2025-12-08T11:34:25.283424Z
  dataset: fineweb_dutch
  input_vectors: 10000000
  method: sc
  dimensionality: 2000
  rank: [150, 150, 150]
  max_iterations: 2000
  iterations: 822
  final_error: 0.9610491394996643
  final_fitness: 0.025616008843949555
  tolerance: 1e-06
  random_state: 1
 

In [10]:
for dim in [1000, 1500, 2000]: #5000]:
    for method in ["counting", "sc"]:
        try:
            tensor = TuckerDecomposition.load_from_disk(
                dataset="fineweb_sparse",
                method=method,
                dims=dim,
                rank=50,
                iterations=10000,
                map_location="cpu"
            )
            print("10K iterations loaded")
        except:
            tensor = TuckerDecomposition.load_from_disk(
                dataset="fineweb_sparse",
                method=method,
                dims=dim,
                rank=150,
                iterations=1000,
                map_location="cpu"
            )
            print("1K iterations loaded")
        sentences_subset = []
        sentence_tuples_subset = []
        for (tuple, sent) in zip(sentence_tuples, sentences):
            try:
                tensor.fetch_latents(tuple)
                sentences_subset.append(sent)
                sentence_tuples_subset.append(tuple)
            except KeyError as e:
                continue
        print(f"Dimension: {dim}, Method: {method} - Using {len(sentences_subset)} sentences out of {len(sentences)}")
        embeddings = model.encode(sentences_subset)
        df_unique = unique_pairwise_cosine(embeddings, labels=sentences_subset)
        projections = [tensor.contribution_tensor(sent_tuple) for sent_tuple in sentence_tuples_subset]
        df_proj_unique = unique_pairwise_cosine_tucker(projections, labels=sentences)
        merged = pd.merge(
            df_unique,
            df_proj_unique,
            on=["i", "j"],
            suffixes=("_sbert", f"_{method}_tucker")
        )
        corr, p_value = spearmanr(merged["similarity_sbert"], merged[f"similarity_{method}_tucker"])
        print(f"Dimension: {dim}, Method: {method} - Spearman correlation: {corr}, p-value: {p_value}")

10K iterations loaded
Dimension: 1000, Method: counting - Using 9 sentences out of 17
Dimension: 1000, Method: counting - Spearman correlation: -0.1702135522304181, p-value: 0.6382593583672791
10K iterations loaded
Dimension: 1000, Method: sc - Using 9 sentences out of 17
Dimension: 1000, Method: sc - Spearman correlation: 0.5272727272727272, p-value: 0.11730806555020223
10K iterations loaded
Dimension: 1500, Method: counting - Using 13 sentences out of 17
Dimension: 1500, Method: counting - Spearman correlation: 0.03175230566534914, p-value: 0.8359636446480664
10K iterations loaded
Dimension: 1500, Method: sc - Using 13 sentences out of 17
Dimension: 1500, Method: sc - Spearman correlation: -0.24126570831949712, p-value: 0.11035443594322443
10K iterations loaded
Dimension: 2000, Method: counting - Using 13 sentences out of 17
Dimension: 2000, Method: counting - Spearman correlation: 0.1976284584980237, p-value: 0.19315450365256714
10K iterations loaded
Dimension: 2000, Method: sc - Us

# Old code
Tries to implement stopping criterium based on semantic similarity

In [21]:
# we import the id, vector, sentence mapping from data/opensubtitles_nl_vectors_ids.csv
import csv
id2sentence = {}
id2vector = {}
with open(DATA_DIR/"opensubtitles_nl_vectors_ids.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        sent_id = row[0]
        vector = row[1]
        # vector is a string representation of a tuple, we convert it back to tuple
        vector = tuple(vector.strip("()").replace("'", "").split(", "))
        sentence = row[2]
        id2sentence[sent_id] = sentence
        id2vector[sent_id] = vector

FileNotFoundError: [Errno 2] No such file or directory: '/home/local/stefa/data/opensubtitles_nl_vectors_ids.csv'

In [11]:
from math import log
from collections import Counter
p_x = Counter() # estimate from counts
p_y = Counter()
p_z = Counter()
p_xy = Counter()
p_yz = Counter()
p_xz = Counter()
p_xyz = Counter()
def specific_interaction_information(x, y, z):
    if min(p_x[x], p_y[y], p_z[z], p_xy[(x,y)], p_yz[(y,z)], p_xz[(x,z)], p_xyz[(x,y,z)]) == 0:
        raise ValueError('One of the probabilities is zero, cannot compute SI1')
    sii = log(
        (p_xy[(x,y)] * p_yz[(y,z)] * p_xz[(x,z)])
        /
        (p_x[x] * p_y[y] * p_z[z] * p_xyz[(x,y,z)])
    )
    return sii

def specific_correlation(x, y, z):
    if min(p_x[x], p_y[y], p_z[z], p_xyz[(x,y,z)]) == 0:
        raise ValueError('One of the probabilities is zero, cannot compute SI1')
    sc = log(
        p_xyz[(x,y,z)]
        /
        (p_x[x] * p_y[y] * p_z[z])
    )
    return sc
# first we build the vocabularies and the clean vectors
clean_vectors = {}
for id, vector in id2vector.items():
    v = vector[0] if vector[0] != "" else "~"
    s = vector[1] if vector[1] != "" else "~"
    o = vector[2] if vector[2] != "" else "~"
    clean_vectors[id] = (v, s, o)

# we build up the counts
total_len = len(clean_vectors)
for v, s, o in clean_vectors.values():
    p_x[v] += 1 / total_len
    p_y[s] += 1 / total_len
    p_z[o] += 1 / total_len
    p_xy[(v,s)] += 1 / total_len
    p_yz[(s,o)] += 1 / total_len
    p_xz[(v,o)] += 1 / total_len
    p_xyz[(v,s,o)] += 1 / total_len
print("Counts built")


Counts built


In [12]:
# we consider only the top N elements in each dimension
k = 750
id_selection = set()
m_c_v = [a for (a, b) in p_x.most_common(k)]
m_c_s = [a for (a, b) in p_y.most_common(k)]
m_c_o = [a for (a, b) in p_z.most_common(k)]
subset_t = []
for id, vector in clean_vectors.items():
    if vector[0] in m_c_v and vector[1] in m_c_s and vector[2] in m_c_o:
        subset_t.append(vector[:3])
        id_selection.add(id)

print(len(subset_t), len(id_selection))

# 1) Build vocabularies in frequency order (stable & useful later)
vocab_v = [v for v, _ in Counter([v for v, _, _ in subset_t]).most_common()]
vocab_s = [s for s, _ in Counter([s for _, s, _ in subset_t]).most_common()]
vocab_o = [o for o, _ in Counter([o for _, _, o in subset_t]).most_common()]

V, S, O = len(vocab_v), len(vocab_s), len(vocab_o)

v2i = {v: i for i, v in enumerate(vocab_v)}
s2i = {s: i for i, s in enumerate(vocab_s)}
o2i = {o: i for i, o in enumerate(vocab_o)}
print(f"Tensor dims: V={V}, S={S}, O={O}")
print(f"Total size: {V * S * O} entries (~{(V * S * O * 4) / (1024 ** 3):.2f} GB as float32)")
print()

769079 769079
Tensor dims: V=750, S=750, O=750
Total size: 421875000 entries (~1.57 GB as float32)



In [13]:
# we split into a train and test set, taking 90% for training
# as it is already shuffled, we just take the top 90% from both subset_t and id_selection
split_index = int(0.95 * len(subset_t))
train_tuples = subset_t[:split_index]
test_tuples = subset_t[split_index:]
train_ids = list(id_selection)[:split_index]
test_ids = list(id_selection)[split_index:]
print(f"Train size: {len(train_tuples)}, Test size: {len(test_tuples)}")
print(f"Train IDs size: {len(train_ids)}, Test IDs size: {len(test_ids)}")

Train size: 730625, Test size: 38454
Train IDs size: 730625, Test IDs size: 38454


In [14]:
import numpy as np
tensor = np.zeros((V, S, O), dtype=np.float32)
sii_tensor = np.zeros((V, S, O), dtype=np.float32)
sc_tensor = np.zeros((V, S, O), dtype=np.float32)
for v, s, o in train_tuples:
    vi = v2i[v]
    si = s2i[s]
    oi = o2i[o]
    tensor[vi, si, oi] += 1 # regular counting
    sii_tensor[vi, si, oi] = specific_interaction_information(v, s, o) # SI1 0_population -> First version summed here for each occurence
    sc_tensor[vi, si, oi] = specific_correlation(v, s, o) # SI2 0_population -> First version summed here for each occurence

In [15]:
len(test_ids)

38454

In [16]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
test_sentences = [id2sentence[sent_id] for sent_id in test_ids]
test_tuples = [clean_vectors[sent_id] for sent_id in test_ids]
test_embeddings = model.encode(test_sentences)
emb = test_embeddings.astype(np.float32, copy=False)

k = 100  # keep top 100 neighbors per sentence
neighbors_idx = np.zeros((emb.shape[0], k), dtype=np.int32)
neighbors_sim = np.zeros((emb.shape[0], k), dtype=np.float32)

batch = 2048  # tune this for your GPU/CPU RAM

for start in tqdm(range(0, emb.shape[0], batch)):
    end = min(start + batch, emb.shape[0])

    # cosine sim = dot product because we normalize first
    sims_block = np.matmul(emb[start:end], emb.T)  # shape (B, N)

    # remove self-match = 1.0 on the diagonal, so it doesn't always win
    for i in range(start, end):
        sims_block[i-start, i] = -np.inf

    # argpartition to get top-k per row without full sort
    idx_part = np.argpartition(-sims_block, k, axis=1)[:, :k]

    # gather sims for those idx
    sims_top = np.take_along_axis(sims_block, idx_part, axis=1)

    # now sort those k to get them ordered
    order = np.argsort(-sims_top, axis=1)
    idx_sorted = np.take_along_axis(idx_part, order, axis=1)
    sims_sorted = np.take_along_axis(sims_top, order, axis=1)

    neighbors_idx[start:end] = idx_sorted
    neighbors_sim[start:end] = sims_sorted

# save compact neighbor graph instead of giant matrix
np.save(DATA_DIR/"train_test/nn_indices.npy", neighbors_idx)
np.save(DATA_DIR/"train_test/nn_sims.npy", neighbors_sim)

100%|██████████| 19/19 [00:13<00:00,  1.40it/s]


In [17]:
# we save this in data/train_test
## remember this uses the dataset loaded in from data/opensubtitles_nl_vectors_ids.csv

import pickle
with open(DATA_DIR/"train_test/train_tuples.pkl", "wb") as f:
    pickle.dump(train_tuples, f)
with open(DATA_DIR/"train_test/test_tuples.pkl", "wb") as f:
    pickle.dump(test_tuples, f)
with open(DATA_DIR/"train_test/train_ids.pkl", "wb") as f:
    pickle.dump(train_ids, f)
with open(DATA_DIR/"train_test/test_ids.pkl", "wb") as f:
    pickle.dump(test_ids, f)
# we save the respective vocabularies as well
vocab = {
    "vocab_v": vocab_v,
    "vocab_s": vocab_s,
    "vocab_o": vocab_o,
    "v2i": v2i,
    "s2i": s2i,
    "o2i": o2i
}
with open(DATA_DIR/"train_test/vocabularies.pkl", "wb") as f:
    pickle.dump(vocab, f)

# we also save the tensors
# we make sure they are torch tensors
tensor_counting = torch.as_tensor(tensor, dtype=torch.float32, device='cpu')
tensor_sii = torch.as_tensor(sii_tensor, dtype=torch.float32, device='cpu')
tensor_sc = torch.as_tensor(sc_tensor, dtype=torch.float32, device='cpu')
with open(DATA_DIR/"train_test/tensor_counting.pkl", "wb") as f:
    pickle.dump(tensor, f)
with open(DATA_DIR/"train_test/tensor_sii.pkl", "wb") as f:
    pickle.dump(sii_tensor, f)
with open(DATA_DIR/"train_test/tensor_sc.pkl", "wb") as f:
    pickle.dump(sc_tensor, f)

In [18]:
# we perform Tucker Decomposition using tensorly
import tensorly as tl
from tensorly.decomposition import tucker, non_negative_tucker, non_negative_tucker_hals
import time, os

tl.set_backend('pytorch')
# torch.cuda.set_device(2)
# ensure float32 and move to GPU (if they are numpy arrays)
tensor_counting = torch.as_tensor(tensor, dtype=torch.float32, device='cuda')
tensor_sc = torch.as_tensor(sc_tensor, dtype=torch.float32, device='cuda')
tensor_sii = torch.as_tensor(sii_tensor, dtype=torch.float32, device='cuda')

In [19]:
import inspect
import sys, io, re
from contextlib import contextmanager


def decompose(tensor, decomp_func=None, non_negative=True, hals=True, rank=None, **kwargs):
    if rank is None:
        rank = [100, 100, 100]
    tic = time.time()
    if not decomp_func:
        if not non_negative:
            decomp = tucker
        elif hals:
            decomp = non_negative_tucker_hals
        else:
            decomp = non_negative_tucker
    else:
        decomp = decomp_func

    # figure out what kwargs this function actually accepts
    sig = inspect.signature(decomp)
    allowed_params = sig.parameters.keys()

    # keep only the kwargs that are valid for this specific function
    filtered_kwargs = {k: v for k, v in kwargs.items() if k in allowed_params}

    core, factors = decomp(tensor, rank=rank, **filtered_kwargs)
    toc = time.time()
    print("Time taken for decomposition", toc - tic)
    return core, factors


# some shenanigans to show iteration numbers live


_NUM = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?"


class IterPrefixer(io.TextIOBase):
    def __init__(self, orig, on_iter=None, autoflush=True):
        self.orig = orig
        self.buf = ""
        self.iter = 0
        self.on_iter = on_iter
        self.autoflush = autoflush
        # NOTE: no lookahead at the end; we'll strip punctuation when parsing
        self._regex = re.compile(
            rf"reconstruction\s*error\s*=\s*({_NUM})\s*,\s*variation\s*=\s*({_NUM})",
            re.IGNORECASE,
        )

    def write(self, s):
        self.buf += s
        # robust line handling (supports mixed \n / \r\n / partial lines)
        while True:
            nl = self.buf.find("\n")
            if nl == -1:
                break
            line = self.buf[:nl]
            self.buf = self.buf[nl + 1:]

            m = self._regex.search(line)
            if m:
                self.iter += 1
                err_str = m.group(1).rstrip(".,;: ")
                var_str = m.group(2).rstrip(".,;: ")
                try:
                    err = float(err_str)
                    var = float(var_str)
                except ValueError:
                    err = var = None  # fail-safe
                if self.on_iter and (err is not None) and (var is not None):
                    try:
                        self.on_iter(self.iter, err, var)
                    except Exception:
                        pass
                line = f"[iter {self.iter:03d}] " + line

            self.orig.write(line + "\n")
            if self.autoflush:
                try:
                    self.orig.flush()
                except Exception:
                    pass
        return len(s)

    def flush(self):
        if self.buf:
            # write any trailing partial line as-is
            self.orig.write(self.buf)
            self.buf = "\r"
        try:
            self.orig.flush()
        except Exception:
            pass


@contextmanager
def live_iteration_numbers(on_iter=None):
    old_stdout, old_stderr = sys.stdout, sys.stderr
    pref_out = IterPrefixer(old_stdout, on_iter=on_iter, autoflush=True)
    pref_err = IterPrefixer(old_stderr, on_iter=on_iter, autoflush=True)
    sys.stdout, sys.stderr = pref_out, pref_err
    try:
        yield
    finally:
        sys.stdout, sys.stderr = old_stdout, old_stderr

# Adapting the function to use this information
We will copy and adapt the non_negative_tucker function and adapt it to our needs
We make sure the validation looks at similarity
! Mind the size of the eval!

In [20]:
with live_iteration_numbers():
    n_iter_max = 100
    rank = [100, 100, 100]
    tol = 1e-12
    verbose = False
    (core_a, factors_a), errors_a = decompose(
        tensor_counting, non_negative=True, hals=False,
        rank=rank, n_iter_max=n_iter_max, init="random", tol=tol, verbose=verbose, return_errors=True
    )

Time taken for decomposition 9.883894443511963


In [22]:
validation_ctx_sample = {
    'nn_indices': np.load(DATA_DIR/"train_test/nn_indices.npy"),
    'nn_sims': np.load(DATA_DIR/"train_test/nn_sims.npy")
}

# Implementing random sampling from the test set for faster evaluation

In [30]:
from tensorly.tenalg import mode_dot
from tensorly.base import unfold
from tensorly.tucker_tensor import tucker_to_tensor, tucker_normalize, validate_tucker_rank
from tensorly.decomposition._tucker import initialize_tucker, TuckerTensor
from scipy.stats import pearsonr
def core_projection(core, factors, test_tuple, weighted=True):
    verb_idx = v2i[test_tuple[0]]
    subj_idx = s2i[test_tuple[1]]
    obj_idx = o2i[test_tuple[2]]
    factor_verb = factors[0].cpu().numpy()
    factor_subj = factors[1].cpu().numpy()
    factor_obj = factors[2].cpu().numpy()
    core = core.cpu().numpy()
    v = factor_verb[verb_idx, :]
    s = factor_subj[subj_idx, :]
    o = factor_obj[obj_idx, :]
    if weighted:
        # elementwise multiply the core by the outer-product weights
        # shape (R1,R2,R3): v[i]*s[j]*o[k] * core[i,j,k]
        proj = np.einsum('i,j,k,ijk->ijk', v, s, o, core)
    else:
        proj = np.einsum('i,j,k->ijk', v, s, o)
    return proj

def eval_sem_alignment(core, factors, validation_ctx, sample_k=None, batch=2048, random_state=42):
    nn_indices_all = validation_ctx['nn_indices']  # (N_eval, K) [global corpus idx]
    nn_sims_all    = validation_ctx['nn_sims']     # (N_eval, K)

    n_eval_total = nn_sims_all.shape[0]
    if (sample_k is None) or (sample_k > n_eval_total):
        # fall back to all items if k not given or too large
        eval_ids = np.arange(n_eval_total, dtype=np.int64)
        nn_indices = nn_indices_all
        nn_sims    = nn_sims_all
    else:
        # random sample k items
        rng = np.random.default_rng(seed=random_state)  # fixed seed for reproducibility
        eval_ids = rng.choice(n_eval_total, size=sample_k, replace=False).astype(np.int64)
        nn_indices = nn_indices_all[eval_ids]
        nn_sims    = nn_sims_all[eval_ids]

    # Collect unique test-set indices needed: the k queries + all their neighbors
    needed_ids = np.unique(np.concatenate([eval_ids, nn_indices.reshape(-1)]))
    # map original test-set indices -> compact 0..M-1
    id_map = {old: i for i, old in enumerate(needed_ids)}
    remapped_eval = np.vectorize(id_map.get)(eval_ids)
    remapped_nn   = np.vectorize(id_map.get)(nn_indices)

    # Build core-projection vectors ONLY for needed_ids
    first_proj = core_projection(core, factors, test_tuples[needed_ids[0]], weighted=True).ravel()
    D = first_proj.size
    P = np.empty((needed_ids.size, D), dtype=np.float32)
    P[0] = first_proj
    for i, gid in enumerate(needed_ids[1:], start=1):
        P[i] = core_projection(core, factors, test_tuples[gid], weighted=True).ravel()

    # Normalize rows in-place for cosine via dot
    norms = np.linalg.norm(P, axis=1, keepdims=True)
    np.divide(P, norms, out=P, where=norms != 0)

    # Compute projected similarities for the sampled eval rows
    proj_sims = np.zeros_like(nn_sims, dtype=np.float32)
    for start in range(0, eval_ids.size, batch):
        end = min(start + batch, eval_ids.size)
        P_q   = P[remapped_eval[start:end]]        # (B, D)
        P_nns = P[remapped_nn[start:end]]          # (B, K, D)
        # cosine = dot because rows are normalized
        proj_sims[start:end] = np.einsum('bd,bkd->bk', P_q, P_nns, optimize=True)

    corr, _ = pearsonr(proj_sims.ravel(), nn_sims.ravel())
    return corr

def non_negative_tucker_custom(
    tensor,
    rank,
    n_iter_max=10,
    init="svd",
    tol=10e-5,
    random_state=None,
    verbose=False,
    return_errors=False,
    normalize_factors=False,
    validation_ctx=None,
    sample_k=None,
    sem_check_every=1
):
    """Non-negative Tucker decomposition

        Iterative multiplicative update, see [2]_

    ~~~~CUSTOM!~~~~
    Now also tracks an external semantic alignment score during training
    if `validation_ctx` is provided. This measures how well the learned
    Tucker factors/Core preserve transformer-based sentence similarity.

    Parameters
    ----------
    tensor : ``ndarray``
    rank : None, int or int list
        size of the core tensor, ``(len(ranks) == tensor.ndim)``
        if int, the same rank is used for all modes
    n_iter_max : int
        maximum number of iteration
    init : {'svd', 'random'} -
        svd results in faster convergence but costly for large tensors
    tol : float
        tolerance to declare convergence
    random_state : {None, int, np.random.RandomState}
        random state for eval sampling
    verbose : int , optional
        level of verbosity
    return_errors : boolean
        Indicates whether the algorithm should return all reconstruction errors
        and computation time of each iteration or not
        Default: False
    normalize_factors : if True, aggregates the norms of the factors in the core.
    validation_ctx : dict or None
        If provided, enables semantic alignment scoring.
    sample_k : int or None
        Subsample size for semantic eval.
    sem_check_every : int or None
        - If int >= 1: compute semantic score every `sem_check_every` iterations
          (and on the final iteration if it didn't land exactly on a multiple).
        - If 0 or None: disable semantic scoring even if `validation_ctx` is set.

    Returns
    -------
    core : ndarray
            positive core of the Tucker decomposition
            has shape `ranks`
    factors : ndarray list
            list of factors of the CP decomposition
            element `i` is of shape ``(tensor.shape[i], rank)``

    errors or metrics : list or dict
        If return_errors is True:
            - if validation_ctx is None: rec_errors (list of floats)
            - else: dict with keys "rec_errors" and "sem_scores"

    References
    ----------
    .. [2] Yong-Deok Kim and Seungjin Choi,
       "Non-negative tucker decomposition",
       IEEE Conference on Computer Vision and Pattern Recognition s(CVPR),
       pp 1-8, 2007

       OG by Tensorly team!
    """



    rank = validate_tucker_rank(tl.shape(tensor), rank=rank)
    print("validated rank", rank)
    epsilon = 10e-12

    # Initialisation
    nn_core, nn_factors = initialize_tucker(
        tensor,
        rank,
        range(tl.ndim(tensor)),
        init=init,
        random_state=random_state,
        non_negative=True,
    )
    print("initialized tucker")
    norm_tensor = tl.norm(tensor, 2)
    print("normalized tensor")
    rec_errors = []
    # NEW: semantic similarity score tracking
    sem_scores = []

    for iteration in range(n_iter_max):
        print(iteration)
        # update for each factor (mode)
        for mode in range(tl.ndim(tensor)):
            print("mode", mode)
            B = tucker_to_tensor((nn_core, nn_factors), skip_factor=mode)
            B = tl.transpose(unfold(B, mode))

            numerator = tl.dot(unfold(tensor, mode), B)
            numerator = tl.clip(numerator, a_min=epsilon, a_max=None)
            denominator = tl.dot(nn_factors[mode], tl.dot(tl.transpose(B), B))
            denominator = tl.clip(denominator, a_min=epsilon, a_max=None)
            nn_factors[mode] *= numerator / denominator

        # update the core
        print("updating the core")
        numerator = tucker_to_tensor((tensor, nn_factors), transpose_factors=True)
        print("tucker to tensor done")
        numerator = tl.clip(numerator, a_min=epsilon, a_max=None)
        for i, f in enumerate(nn_factors):
            print("factor", i)
            if i:
                denominator = mode_dot(denominator, tl.dot(tl.transpose(f), f), i)
            else:
                denominator = mode_dot(nn_core, tl.dot(tl.transpose(f), f), i)
        denominator = tl.clip(denominator, a_min=epsilon, a_max=None)
        print("denominator done")
        nn_core *= numerator / denominator

        # reconstruction error
        print("getting reconstruction error")
        rec_error = (
            tl.norm(tensor - tucker_to_tensor((nn_core, nn_factors)), 2) / norm_tensor
        )
        rec_errors.append(rec_error)

        # ---- NEW: semantic external validation ----
        if validation_ctx is not None:
            sem_score = eval_sem_alignment(nn_core, nn_factors, validation_ctx, sample_k=sample_k)
            sem_scores.append(sem_score)
        else:
            sem_score = None

        if iteration > 1 and verbose:
            msg = (
                f"reconstruction error={rec_errors[-1]}, "
                f"variation={rec_errors[-2] - rec_errors[-1]}"
            )
            if sem_score is not None:
                msg += f", sem_score={sem_score}"
            print(msg + ".")

        if iteration > 1 and tl.abs(rec_errors[-2] - rec_errors[-1]) < tol:
            if verbose:
                print(f"converged in {iteration} iterations.")
            break
        if normalize_factors:
            nn_core, nn_factors = tucker_normalize((nn_core, nn_factors))

    tensor = TuckerTensor((nn_core, nn_factors))
    if return_errors:
        # keep backward compatibility if no validation_ctx
        if validation_ctx is None:
            return tensor, rec_errors
        else:
            return tensor, {"rec_errors": rec_errors, "sem_scores": sem_scores}
    else:
        return tensor


In [32]:
with live_iteration_numbers():
    n_iter_max = 3
    rank = [100, 100, 100]
    tol = 1e-12
    verbose = True
    (core_a, factors_a), errors_a = non_negative_tucker_custom(
        tensor_counting,
        rank=rank, n_iter_max=n_iter_max, init="svd", tol=tol,
        verbose=verbose, return_errors=True,
        validation_ctx=validation_ctx_sample,
    )

validated rank [100, 100, 100]
initialized tucker
normalized tensor
0
mode 0
mode 1
mode 2
updating the core


KeyboardInterrupt: 

In [53]:
errors_a

{'rec_errors': [tensor(0.4413, device='cuda:0'),
  tensor(0.3531, device='cuda:0'),
  tensor(0.2922, device='cuda:0')],
 'sem_scores': [np.float32(0.071959965),
  np.float32(0.16851804),
  np.float32(-0.028991021)]}

In [27]:
errors_a

NameError: name 'errors_a' is not defined